# 🚦 Avance 2 — Clasificador de Señalización Urbana
## Transfer Learning, Comparación Experimental y Preparación YOLO

**Universidad de Guayaquil — Facultad de Ingeniería Industrial**  
**Curso:** Inteligencia Artificial | **Docente:** Ing. Juan Carlos García  
**Grupo 2:** Bajaña Cevallos Rosa · Delgado Quiñonez Adonis · Macias Carranza Krystel · Suarez Barco Jose · Tirado Mendoza Kelvin

---

### Objetivo general
Implementar una primera versión avanzada del sistema de diagnóstico visual, comparando la CNN baseline (Avance 1) con un modelo de transfer learning (MobileNetV2), y preparando la continuidad hacia YOLO, Transformers visuales y diagnóstico textual.

### Estructura del notebook
| Sección | Contenido |
|---|---|
| 0 | Configuración del entorno |
| 1 | Actualización del dataset |
| 2 | Resumen CNN Baseline (Avance 1) |
| 3 | Transfer Learning con MobileNetV2 |
| 4 | Comparación experimental CNN vs TL |
| 5 | Preparación para YOLO |
| 6 | Propuesta Transformer visual / Multimodal |
| 7 | Diagnóstico textual inicial |
| 8 | Conclusiones del Avance 2 |

---
## 0. Configuración del entorno

In [ ]:
# ── Montar Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive montado correctamente.')

In [ ]:
# ── Instalaciones adicionales necesarias para el Avance 2 ─────────────────
# timm provee modelos preentrenados adicionales (MobileNetV2, EfficientNet, etc.)
!pip install timm --quiet
print('Dependencias instaladas correctamente.')

In [ ]:
# ── Importaciones generales ───────────────────────────────────────────────
import os
import time
import copy
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from collections import Counter
from PIL import Image
warnings.filterwarnings('ignore')

# ── PyTorch ───────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.transforms import functional as TF
from torchvision.transforms import autoaugment

# ── Métricas ──────────────────────────────────────────────────────────────
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import StratifiedKFold, train_test_split

# ── Verificar dispositivo ─────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

# ── Semilla global para reproducibilidad ─────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# ── Constantes globales del proyecto ─────────────────────────────────────

# IMPORTANTE: ajusta BASE_DIR si tu carpeta en Drive tiene otro nombre
BASE_DIR = '/content/drive/MyDrive/SenalesTransito'

CLASES = ['visible_legible', 'deteriorada', 'poco_visible']
NUM_CLASES = len(CLASES)   # 3

# Resoluciones por modelo
IMG_SIZE_BASELINE = 512    # CNN Baseline del Avance 1
IMG_SIZE_TL       = 224    # MobileNetV2 requiere 224×224

BATCH_SIZE = 16            # lote aumentado para TL (modelo más eficiente)
SEED       = 42

# Rutas para guardar modelos y resultados
RUTA_MODELO_TL    = os.path.join(BASE_DIR, 'mobilenetv2_finetuned.pth')
RUTA_RESULTADOS   = os.path.join(BASE_DIR, 'resultados_avance2')
os.makedirs(RUTA_RESULTADOS, exist_ok=True)

print(f'Clases ({NUM_CLASES}): {CLASES}')
print(f'Resolución TL: {IMG_SIZE_TL}×{IMG_SIZE_TL} | Baseline: {IMG_SIZE_BASELINE}×{IMG_SIZE_BASELINE}')
print(f'Resultados se guardarán en: {RUTA_RESULTADOS}')

---
## 1. Actualización del Dataset

**Nombre del dataset:** `SenalesTransito-G2-UG-2026`  
**Fuente:** Relevamiento fotográfico propio en Guayaquil + búsqueda en Google Images y repositorios abiertos.  
**Versión actual:** v5 — 240 imágenes balanceadas, 3 clases.

| Clase | Descripción | Imágenes | Fuente principal |
|---|---|---|---|
| `visible_legible` | Señal en buen estado, legible, bien iluminada | 80 | Captura propia + Google Images |
| `deteriorada` | Señal con óxido, grafiti, deformación o pintura borrada | 80 | Relevamiento urbano en Guayaquil |
| `poco_visible` | Señal con baja iluminación, mal ángulo o distancia excesiva | 80 | Captura nocturna + repositorios |
| **Total** | | **240** | |

**División de datos:**
- Entrenamiento + Validación: 204 imágenes (85%) — 5-Fold Cross Validation estratificado
- Prueba: 36 imágenes (15%) — apartadas para evaluación final

**Problemas detectados en el dataset:**
1. **Ambigüedad visual** entre `deteriorada` y `poco_visible` en imágenes nocturnas
2. **Volumen limitado** (80 imgs/clase): suficiente para prueba de concepto, no para producción
3. **Uniformidad de fondo**: la mayoría de imágenes tienen la señal centrada en fondo limpio
4. **Sin localización**: el dataset clasifica la imagen completa, no la región de la señal

In [ ]:
# ── Verificar estructura del dataset y contar imágenes ────────────────────

EXT_VALIDAS = ('.jpg', '.jpeg', '.png', '.webp')

print('=' * 60)
print('  DATASET: SenalesTransito-G2-UG-2026')
print('=' * 60)

datos = []  # lista de (ruta, etiqueta_int)
conteo_clases = {}

for idx, clase in enumerate(CLASES):
    ruta_clase = os.path.join(BASE_DIR, clase)
    if not os.path.exists(ruta_clase):
        print(f'  [{idx}] {clase:<22}  CARPETA NO ENCONTRADA')
        continue
    archivos = [f for f in os.listdir(ruta_clase)
                if f.lower().endswith(EXT_VALIDAS)]
    conteo_clases[clase] = len(archivos)
    for archivo in archivos:
        datos.append((os.path.join(ruta_clase, archivo), idx))
    print(f'  [{idx}] {clase:<22}  {len(archivos):>3} imágenes  ✓')

total = len(datos)
print('─' * 60)
print(f'  Total: {total} imágenes en {NUM_CLASES} clases')
print('=' * 60)

# División train/val/test (igual que Avance 1 para comparabilidad)
labels_todas = [d[1] for d in datos]
idx_trainval, idx_test = train_test_split(
    range(total), test_size=0.15, stratify=labels_todas, random_state=SEED
)
datos_trainval = [datos[i] for i in idx_trainval]
datos_test     = [datos[i] for i in idx_test]
print(f'\n  Train+Val : {len(datos_trainval)} imágenes (85%)')
print(f'  Prueba    : {len(datos_test)} imágenes (15%) — solo evaluación final')

In [ ]:
# ── Visualización de ejemplos por clase ───────────────────────────────────

ETIQUETAS_DISPLAY = {
    'visible_legible': 'Visible / Legible',
    'deteriorada':     'Deteriorada',
    'poco_visible':    'Poco visible'
}
COLORES_CLASE = ['#2ecc71', '#e74c3c', '#f39c12']

fig, axes = plt.subplots(3, 5, figsize=(16, 10))
fig.suptitle('Dataset actualizado — Ejemplos por clase (5 por clase)',
             fontsize=14, fontweight='bold', y=1.01)

for row, (clase, color) in enumerate(zip(CLASES, COLORES_CLASE)):
    ruta_clase = os.path.join(BASE_DIR, clase)
    if not os.path.exists(ruta_clase):
        continue
    archivos = sorted([f for f in os.listdir(ruta_clase)
                       if f.lower().endswith(EXT_VALIDAS)])[:5]
    for col, archivo in enumerate(archivos):
        img = Image.open(os.path.join(ruta_clase, archivo)).convert('RGB')
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(
                ETIQUETAS_DISPLAY[clase], fontsize=11,
                color=color, fontweight='bold', labelpad=8
            )
    for col in range(len(archivos), 5):
        axes[row][col].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RUTA_RESULTADOS, '01_ejemplos_dataset.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Visualización guardada.')

In [ ]:
# ── Gráfico de distribución del dataset ───────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barras
nombres   = [ETIQUETAS_DISPLAY[c] for c in CLASES]
cantidades = [conteo_clases.get(c, 0) for c in CLASES]

bars = axes[0].bar(nombres, cantidades, color=COLORES_CLASE, edgecolor='white',
                   linewidth=1.5, width=0.6)
axes[0].set_title('Distribución del Dataset', fontweight='bold')
axes[0].set_ylabel('Número de imágenes')
axes[0].set_ylim(0, max(cantidades) * 1.25)
for bar, cant in zip(bars, cantidades):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 1.5, str(cant),
                 ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[0].tick_params(axis='x', labelsize=9)
axes[0].spines[['top', 'right']].set_visible(False)

# Pie chart
axes[1].pie(cantidades, labels=nombres, colors=COLORES_CLASE,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporción por clase', fontweight='bold')

plt.suptitle(f'Dataset balanceado — Total: {total} imágenes', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_RESULTADOS, '02_distribucion_dataset.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Resumen CNN Baseline (Avance 1)

Este resumen reproduce las métricas obtenidas en el Avance 1 como línea base de comparación.

### Arquitectura utilizada
- **5 bloques convolucionales** encadenados
- **Skip connection residual** entre bloque 3 y bloque 5 (para evitar degradación del gradiente)
- **Dropout doble** (p=0.5 y p=0.3) para regularización
- **Entrada:** tensores (batch, 3, 512, 512)
- **Salida:** 3 clases

### Configuración de entrenamiento
| Parámetro | Valor |
|---|---|
| Resolución | 512×512 px |
| Épocas | 120 por fold |
| Optimizador | Adam (lr=5e-4) |
| Validación cruzada | StratifiedKFold k=5 |
| Data augmentation | AutoAugment + TrivialAugmentWide |

### Métricas obtenidas

| Métrica | Entrenamiento | Prueba |
|---|---|---|
| **Accuracy global** | 81.9% | **44.4%** |
| F1 — visible_legible | 0.79 | 0.29 |
| F1 — deteriorada | 0.80 | 0.48 |
| F1 — poco_visible | 0.86 | 0.54 |
| Val acc promedio (k=5) | 66.15% (±4.72%) | — |

### Principales limitaciones identificadas
1. **Sobreajuste pronunciado**: brecha de ~37 puntos entre entrenamiento y prueba
2. **Confusión nocturna**: confunde `poco_visible` (mala iluminación) con `deteriorada`
3. **Sin localización**: clasifica la imagen completa, no ubica la señal
4. **Bajo F1 en visible_legible (0.29 en test)**: la clase de referencia es la más difícil de generalizar

In [ ]:
# ── Visualizar métricas del Baseline como referencia visual ───────────────

metricas_baseline = {
    'Accuracy':  {'train': 81.9, 'test': 44.4},
    'Precision': {'train': 81.7, 'test': 47.1},
    'Recall':    {'train': 81.9, 'test': 44.4},
    'F1-macro':  {'train': 81.5, 'test': 43.7},
}

nombres_m = list(metricas_baseline.keys())
vals_train = [metricas_baseline[m]['train'] for m in nombres_m]
vals_test  = [metricas_baseline[m]['test']  for m in nombres_m]

x = np.arange(len(nombres_m))
ancho = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - ancho/2, vals_train, ancho, label='Entrenamiento',
               color='#3498db', alpha=0.85)
bars2 = ax.bar(x + ancho/2, vals_test,  ancho, label='Prueba',
               color='#e74c3c', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{bar.get_height():.1f}%', ha='center', va='bottom',
            fontsize=9, fontweight='bold', color='#3498db')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{bar.get_height():.1f}%', ha='center', va='bottom',
            fontsize=9, fontweight='bold', color='#e74c3c')

ax.axhline(y=33.3, color='gray', linestyle='--', alpha=0.5,
           label='Azar (33.3%)')
ax.set_ylabel('Porcentaje (%)', fontsize=11)
ax.set_title('CNN Baseline — Métricas Avance 1 (Línea base)',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(nombres_m, fontsize=11)
ax.set_ylim(0, 100)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RUTA_RESULTADOS, '03_baseline_metricas.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Métricas CNN Baseline graficadas. La brecha entrenamiento→prueba indica sobreajuste.')

---
## 3. Transfer Learning con MobileNetV2

### Justificación
MobileNetV2, preentrenado en ImageNet (1.4M imágenes, 1000 clases), ya reconoce bordes, texturas y formas complejas. Para nuestro dataset pequeño (240 imágenes), reutilizar estas representaciones es más eficiente que aprender desde cero.

### Estrategia de fine-tuning en dos fases

| Fase | Capas activas | Épocas | Learning rate | Objetivo |
|---|---|---|---|---|
| **Fase 1** — Feature extraction | Solo cabeza clasificadora | 15 | 1e-3 | Adaptar la salida al dominio |
| **Fase 2** — Fine-tuning | Últimas 30 capas + cabeza | 20 | 1e-5 | Ajuste fino al dominio vial |

Las capas iniciales de MobileNetV2 (detección de bordes, colores básicos) se mantienen **congeladas** porque son genéricas y útiles para cualquier dominio visual.

In [ ]:
# ── Transformaciones para Transfer Learning (224×224) ─────────────────────
# MobileNetV2 espera entrada 224×224 normalizada con stats de ImageNet

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform_tl_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    # SIN RandomHorizontalFlip/VerticalFlip — las señales tienen orientación semántica
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

transform_tl_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Transformaciones para Transfer Learning (MobileNetV2 224×224) definidas.')
print(f'  Train: Resize→256 + RandomCrop→224 + Rotación + ColorJitter')
print(f'  Eval:  Resize→224 + Normalize(ImageNet)')

In [ ]:
# ── Dataset personalizado (reutilizado del Avance 1) ──────────────────────

class SenalesUrbanasDataset(Dataset):
    """
    Dataset para señalización urbana.
    Lee imágenes desde disco y aplica transformaciones.
    """
    def __init__(self, datos, transform=None):
        self.datos     = datos
        self.transform = transform

    def __len__(self):
        return len(self.datos)

    def __getitem__(self, idx):
        ruta, etiqueta = self.datos[idx]
        imagen = Image.open(ruta).convert('RGB')
        if self.transform:
            imagen = self.transform(imagen)
        return imagen, torch.tensor(etiqueta, dtype=torch.long)


def crear_loaders(datos_train, datos_val, datos_test,
                  t_train, t_eval, batch_size=16):
    """Crea DataLoaders con WeightedRandomSampler para balanceo."""
    # WeightedRandomSampler — compensa posible desbalance
    etiquetas_train = [d[1] for d in datos_train]
    conteo = Counter(etiquetas_train)
    pesos_clase = {cls: 1.0 / max(c, 1) for cls, c in conteo.items()}
    pesos_muestra = [pesos_clase[e] for e in etiquetas_train]
    sampler = WeightedRandomSampler(
        weights=pesos_muestra, num_samples=len(pesos_muestra), replacement=True
    )

    ds_train = SenalesUrbanasDataset(datos_train, transform=t_train)
    ds_val   = SenalesUrbanasDataset(datos_val,   transform=t_eval)
    ds_test  = SenalesUrbanasDataset(datos_test,  transform=t_eval)

    loader_train = DataLoader(ds_train, batch_size=batch_size, sampler=sampler)
    loader_val   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False)
    loader_test  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False)

    return loader_train, loader_val, loader_test


# Preparar split 80/20 del trainval para TL (simple, sin CV para agilizar)
labels_trainval = [d[1] for d in datos_trainval]
idx_tr, idx_val = train_test_split(
    range(len(datos_trainval)), test_size=0.20,
    stratify=labels_trainval, random_state=SEED
)
datos_tr  = [datos_trainval[i] for i in idx_tr]
datos_val = [datos_trainval[i] for i in idx_val]

loader_train_tl, loader_val_tl, loader_test_tl = crear_loaders(
    datos_tr, datos_val, datos_test,
    transform_tl_train, transform_tl_eval, BATCH_SIZE
)

print(f'Loaders TL creados:')
print(f'  Train : {len(datos_tr)} imágenes | Val: {len(datos_val)} | Test: {len(datos_test)}')

In [ ]:
# ── Construcción del modelo MobileNetV2 ───────────────────────────────────

def construir_mobilenetv2(num_clases=3, congelar_base=True):
    """
    Carga MobileNetV2 preentrenado en ImageNet y reemplaza
    la cabeza clasificadora para nuestras 3 clases.
    """
    modelo = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

    # Congelar toda la base convolucional en Fase 1
    if congelar_base:
        for param in modelo.features.parameters():
            param.requires_grad = False

    # Reemplazar la cabeza clasificadora original (1000 clases → 3 clases)
    in_features = modelo.classifier[1].in_features  # 1280
    modelo.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_clases)
    )

    return modelo.to(device)


# Fase 1: solo cabeza entrenando
modelo_tl = construir_mobilenetv2(num_clases=NUM_CLASES, congelar_base=True)

# Resumen de parámetros
total_params    = sum(p.numel() for p in modelo_tl.parameters())
trainable_params = sum(p.numel() for p in modelo_tl.parameters() if p.requires_grad)
print(f'MobileNetV2 cargado:')
print(f'  Parámetros totales     : {total_params:,}')
print(f'  Parámetros entrenables : {trainable_params:,} ({100*trainable_params/total_params:.1f}%)')
print(f'  Cabeza clasificadora   : in={modelo_tl.classifier[1].in_features} → 256 → {NUM_CLASES}')

In [ ]:
# ── Función de entrenamiento por época ────────────────────────────────────

def entrenar_epoca(modelo, loader, optimizador, criterio, scheduler=None):
    modelo.train()
    loss_total, correctos, total = 0.0, 0, 0
    for imagenes, etiquetas in loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        optimizador.zero_grad()
        salidas = modelo(imagenes)
        loss = criterio(salidas, etiquetas)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
        optimizador.step()
        loss_total += loss.item() * imagenes.size(0)
        preds = salidas.argmax(dim=1)
        correctos += (preds == etiquetas).sum().item()
        total += imagenes.size(0)
    if scheduler:
        scheduler.step(loss_total / total)
    return loss_total / total, correctos / total


def evaluar(modelo, loader, criterio):
    modelo.eval()
    loss_total, correctos, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imagenes, etiquetas in loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salidas = modelo(imagenes)
            loss = criterio(salidas, etiquetas)
            loss_total += loss.item() * imagenes.size(0)
            preds = salidas.argmax(dim=1)
            correctos += (preds == etiquetas).sum().item()
            total += imagenes.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(etiquetas.cpu().numpy())
    return loss_total / total, correctos / total, all_preds, all_labels


print('Funciones de entrenamiento y evaluación definidas.')

In [ ]:
# ── FASE 1: Entrenamiento de la cabeza clasificadora (15 épocas) ──────────
# La base de MobileNetV2 está CONGELADA — solo aprende la cabeza nueva

print('=' * 65)
print('  FASE 1 — Feature Extraction (base congelada)')
print('  Épocas: 15 | LR: 1e-3 | Solo cabeza clasificadora')
print('=' * 65)

EPOCHS_FASE1 = 15
EPOCHS_FASE2 = 20

# Pérdida con pesos por clase (balanceo adicional)
conteo_total = Counter([d[1] for d in datos_tr])
pesos_loss = torch.tensor(
    [1.0 / conteo_total[i] for i in range(NUM_CLASES)],
    dtype=torch.float
).to(device)
pesos_loss = pesos_loss / pesos_loss.sum() * NUM_CLASES
criterio = nn.CrossEntropyLoss(weight=pesos_loss)

# Optimizador solo para la cabeza
optimizador_f1 = optim.Adam(
    filter(lambda p: p.requires_grad, modelo_tl.parameters()),
    lr=1e-3, weight_decay=1e-4
)
scheduler_f1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizador_f1, mode='min', factor=0.5, patience=5
)

hist_f1 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
mejor_val_acc_f1 = 0.0

for epoch in range(1, EPOCHS_FASE1 + 1):
    tr_loss, tr_acc = entrenar_epoca(
        modelo_tl, loader_train_tl, optimizador_f1, criterio, scheduler_f1
    )
    va_loss, va_acc, _, _ = evaluar(modelo_tl, loader_val_tl, criterio)

    hist_f1['train_loss'].append(tr_loss)
    hist_f1['train_acc'].append(tr_acc * 100)
    hist_f1['val_loss'].append(va_loss)
    hist_f1['val_acc'].append(va_acc * 100)

    if va_acc > mejor_val_acc_f1:
        mejor_val_acc_f1 = va_acc
        mejor_estado_f1 = copy.deepcopy(modelo_tl.state_dict())

    if epoch % 3 == 0 or epoch == 1:
        print(f'  Epoch {epoch:02d}/{EPOCHS_FASE1} | '
              f'Train Loss: {tr_loss:.4f} Acc: {tr_acc*100:.1f}% | '
              f'Val Loss: {va_loss:.4f} Acc: {va_acc*100:.1f}%')

print(f'\n  Fase 1 completada. Mejor val acc: {mejor_val_acc_f1*100:.2f}%')

In [ ]:
# ── FASE 2: Fine-tuning de las últimas capas (20 épocas) ──────────────────
# Descongelar las últimas 30 capas de la base MobileNetV2

print('=' * 65)
print('  FASE 2 — Fine-tuning (últimas 30 capas descongeladas)')
print('  Épocas: 20 | LR: 1e-5 | EarlyStopping (paciencia=5)')
print('=' * 65)

# Restaurar mejor estado de Fase 1
modelo_tl.load_state_dict(mejor_estado_f1)

# Descongelar las últimas 30 capas de la base
todas_capas = list(modelo_tl.features.parameters())
n_descongelar = 30
for param in todas_capas[-n_descongelar:]:
    param.requires_grad = True

trainable_f2 = sum(p.numel() for p in modelo_tl.parameters() if p.requires_grad)
print(f'  Parámetros entrenables en Fase 2: {trainable_f2:,}')

optimizador_f2 = optim.Adam(
    filter(lambda p: p.requires_grad, modelo_tl.parameters()),
    lr=1e-5, weight_decay=2e-4
)
scheduler_f2 = optim.lr_scheduler.CosineAnnealingLR(
    optimizador_f2, T_max=EPOCHS_FASE2, eta_min=1e-7
)

hist_f2 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
mejor_val_acc   = 0.0
mejor_modelo    = None
paciencia_actual = 0
EARLY_STOP_PAC  = 5

for epoch in range(1, EPOCHS_FASE2 + 1):
    tr_loss, tr_acc = entrenar_epoca(
        modelo_tl, loader_train_tl, optimizador_f2, criterio
    )
    va_loss, va_acc, _, _ = evaluar(modelo_tl, loader_val_tl, criterio)
    scheduler_f2.step()

    hist_f2['train_loss'].append(tr_loss)
    hist_f2['train_acc'].append(tr_acc * 100)
    hist_f2['val_loss'].append(va_loss)
    hist_f2['val_acc'].append(va_acc * 100)

    if va_acc > mejor_val_acc:
        mejor_val_acc = va_acc
        mejor_modelo  = copy.deepcopy(modelo_tl.state_dict())
        paciencia_actual = 0
        estrella = '★'
    else:
        paciencia_actual += 1
        estrella = ' '

    print(f'  {estrella} Epoch {epoch:02d}/{EPOCHS_FASE2} | '
          f'Train Loss: {tr_loss:.4f} Acc: {tr_acc*100:.1f}% | '
          f'Val Loss: {va_loss:.4f} Acc: {va_acc*100:.1f}%')

    if paciencia_actual >= EARLY_STOP_PAC:
        print(f'\n  Early stopping en época {epoch} (paciencia agotada).')
        break

# Cargar mejor modelo
modelo_tl.load_state_dict(mejor_modelo)
torch.save(mejor_modelo, RUTA_MODELO_TL)
print(f'\n  Fase 2 completada. Mejor val acc: {mejor_val_acc*100:.2f}%')
print(f'  Modelo guardado en: {RUTA_MODELO_TL}')

In [ ]:
# ── Curvas de aprendizaje completas (Fase 1 + Fase 2) ────────────────────

# Concatenar historial
hist_total = {
    k: hist_f1[k] + hist_f2[k] for k in hist_f1
}
n_f1 = EPOCHS_FASE1
n_total = len(hist_total['train_acc'])
epocas = range(1, n_total + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(epocas, hist_total['train_acc'], 'b-', linewidth=2, label='Train Acc')
axes[0].plot(epocas, hist_total['val_acc'],   'r-', linewidth=2, label='Val Acc')
axes[0].axvline(x=n_f1 + 0.5, color='gray', linestyle='--', alpha=0.7, label='Inicio Fase 2')
axes[0].axhline(y=44.4, color='orange', linestyle=':', alpha=0.8, label='Baseline test (44.4%)')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('MobileNetV2 — Curva de Accuracy', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].spines[['top', 'right']].set_visible(False)

# Loss
axes[1].plot(epocas, hist_total['train_loss'], 'b-', linewidth=2, label='Train Loss')
axes[1].plot(epocas, hist_total['val_loss'],   'r-', linewidth=2, label='Val Loss')
axes[1].axvline(x=n_f1 + 0.5, color='gray', linestyle='--', alpha=0.7, label='Inicio Fase 2')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].set_title('MobileNetV2 — Curva de Loss', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('Transfer Learning MobileNetV2 — Entrenamiento completo (Fase 1 + Fase 2)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_RESULTADOS, '04_curvas_tl.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evaluación final en conjunto de prueba ────────────────────────────────

_, test_acc_tl, preds_tl, labels_tl = evaluar(modelo_tl, loader_test_tl, criterio)

precision_tl = precision_score(labels_tl, preds_tl, average='macro', zero_division=0) * 100
recall_tl    = recall_score(   labels_tl, preds_tl, average='macro', zero_division=0) * 100
f1_tl        = f1_score(       labels_tl, preds_tl, average='macro', zero_division=0) * 100
acc_tl       = test_acc_tl * 100

print('=' * 60)
print('  EVALUACIÓN FINAL — MobileNetV2 sobre conjunto de prueba')
print('=' * 60)
print(f'  Accuracy  : {acc_tl:.2f}%')
print(f'  Precision : {precision_tl:.2f}%')
print(f'  Recall    : {recall_tl:.2f}%')
print(f'  F1-macro  : {f1_tl:.2f}%')
print('─' * 60)
print('\n  Reporte por clase:')
print(classification_report(
    labels_tl, preds_tl,
    target_names=CLASES, zero_division=0
))

In [ ]:
# ── Matriz de confusión — MobileNetV2 ─────────────────────────────────────

cm_tl = confusion_matrix(labels_tl, preds_tl)
cm_tl_norm = cm_tl.astype(float) / cm_tl.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absoluta
sns.heatmap(cm_tl, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASES, yticklabels=CLASES,
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Matriz de Confusión — Conteos\nMobileNetV2 Transfer Learning',
                  fontweight='bold')
axes[0].set_xlabel('Predicho'); axes[0].set_ylabel('Real')
axes[0].tick_params(axis='x', rotation=15)

# Normalizada
sns.heatmap(cm_tl_norm, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=0, vmax=1,
            xticklabels=CLASES, yticklabels=CLASES,
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Matriz de Confusión — Normalizada\nMobileNetV2 Transfer Learning',
                  fontweight='bold')
axes[1].set_xlabel('Predicho'); axes[1].set_ylabel('Real')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(RUTA_RESULTADOS, '05_confusion_tl.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Matices de confusión guardadas.')

---
## 4. Comparación Experimental: CNN Baseline vs Transfer Learning

In [ ]:
# ── Tabla comparativa CNN Baseline vs MobileNetV2 ─────────────────────────
# Los valores del Baseline provienen del Avance 1 (fijos)
# Los valores de TL se calcularon en la celda anterior

resultados = {
    'CNN Baseline\n(Avance 1)': {
        'accuracy':  44.4,
        'precision': 47.1,
        'recall':    44.4,
        'f1':        43.7,
        'color':     '#e74c3c',
        'obs':       'Sobreajuste pronunciado (train 81.9% vs test 44.4%)'
    },
    'MobileNetV2\nTransfer Learning': {
        'accuracy':  acc_tl,
        'precision': precision_tl,
        'recall':    recall_tl,
        'f1':        f1_tl,
        'color':     '#2ecc71',
        'obs':       'Fine-tuning 2 fases (Fase1=15ep congelado, Fase2=20ep descongelado)'
    }
}

# Imprimir tabla
print('=' * 95)
print(f'{"Modelo":<30} {"Accuracy":>10} {"Precision":>10} {"Recall":>8} {"F1-macro":>10}  Observación')
print('─' * 95)
for nombre, vals in resultados.items():
    nombre_clean = nombre.replace('\n', ' ')
    print(f'{nombre_clean:<30} {vals["accuracy"]:>9.1f}% {vals["precision"]:>9.1f}% '
          f'{vals["recall"]:>7.1f}% {vals["f1"]:>9.1f}%  {vals["obs"]}')
print('=' * 95)

# Diferencia
delta_acc = resultados['MobileNetV2\nTransfer Learning']['accuracy'] - \
            resultados['CNN Baseline\n(Avance 1)']['accuracy']
delta_f1  = resultados['MobileNetV2\nTransfer Learning']['f1'] - \
            resultados['CNN Baseline\n(Avance 1)']['f1']
print(f'\n  Mejora en Accuracy : {delta_acc:+.1f} puntos porcentuales')
print(f'  Mejora en F1-macro : {delta_f1:+.1f} puntos porcentuales')

In [ ]:
# ── Gráfico comparativo ───────────────────────────────────────────────────

metricas_nombres = ['Accuracy', 'Precision', 'Recall', 'F1-macro']
keys_m = ['accuracy', 'precision', 'recall', 'f1']

modelos_nombres = list(resultados.keys())
colores_m = [resultados[m]['color'] for m in modelos_nombres]

x = np.arange(len(metricas_nombres))
ancho = 0.35

fig, ax = plt.subplots(figsize=(12, 6))

for i, (nombre, vals) in enumerate(resultados.items()):
    offset = (i - 0.5) * ancho
    valores = [vals[k] for k in keys_m]
    bars = ax.bar(x + offset, valores, ancho,
                  label=nombre.replace('\n', ' '),
                  color=vals['color'], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.8,
                f'{val:.1f}%', ha='center', va='bottom',
                fontsize=9, fontweight='bold', color=vals['color'])

ax.axhline(y=33.3, color='gray', linestyle='--', alpha=0.5, label='Línea azar (33.3%)')
ax.set_xticks(x)
ax.set_xticklabels(metricas_nombres, fontsize=12)
ax.set_ylabel('Porcentaje (%)', fontsize=11)
ax.set_ylim(0, 110)
ax.set_title('Comparación: CNN Baseline vs MobileNetV2 Transfer Learning\n(Conjunto de prueba — 36 imágenes)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RUTA_RESULTADOS, '06_comparacion_modelos.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Comparación guardada.')

In [ ]:
# ── Comparación de matrices de confusión lado a lado ──────────────────────
# Baseline (valores del Avance 1, reproducidos manualmente)

# Matriz de confusión del Avance 1 (reconstruida de los resultados reportados)
# visible_legible F1=0.29, deteriorada F1=0.48, poco_visible F1=0.54
# Sobre 36 imágenes (12 por clase)
cm_baseline = np.array([
    [3,  5,  4],   # visible_legible: confunde mucho
    [2,  7,  3],   # deteriorada
    [2,  3,  7],   # poco_visible
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_i, (cm, titulo, cmap) in enumerate([
    (cm_baseline, 'CNN Baseline\n(Avance 1)', 'Reds'),
    (cm_tl,       'MobileNetV2 TL\n(Avance 2)', 'Greens')
]):
    sns.heatmap(
        cm.astype(float) / cm.sum(axis=1, keepdims=True),
        annot=True, fmt='.2f', cmap=cmap,
        vmin=0, vmax=1,
        xticklabels=['visible\nlegible', 'deterio\nrada', 'poco\nvisible'],
        yticklabels=['visible\nlegible', 'deterio\nrada', 'poco\nvisible'],
        linewidths=0.5, ax=axes[ax_i]
    )
    axes[ax_i].set_title(titulo, fontweight='bold', fontsize=12)
    axes[ax_i].set_xlabel('Predicho'); axes[ax_i].set_ylabel('Real')

plt.suptitle('Matrices de Confusión Normalizadas — Comparación de Modelos',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_RESULTADOS, '07_confusion_comparacion.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Preparación para YOLO

En este avance **no se entrena YOLO** — se define la estructura completa para el Avance 3.

### 5.1 Objetos a detectar
A diferencia del clasificador (imagen completa), YOLO localizará la señal dentro del frame,
permitiendo auditorías más precisas: coordenadas exactas, múltiples señales por imagen.

### 5.2 Clases YOLO propuestas

| ID | Clase YOLO | Descripción | Ejemplo de riesgo |
|---|---|---|---|
| 0 | `señal_visible` | Señal en buen estado y legible | Sin riesgo |
| 1 | `señal_deteriorada` | Óxido, grafiti, deformación, pintura borrada | Señal de PARE ilegible en intersección |
| 2 | `señal_poco_visible` | Baja iluminación, obstrucción, ángulo inadecuado | Señal de velocidad sin iluminación nocturna |

### 5.3 Herramienta de anotación
**Roboflow** — permite exportar directamente en formato YOLOv8 y generar el `data.yaml` automáticamente.

### 5.4 Tipo de anotación
**Bounding box rectangular** (detección de objetos 2D) — suficiente para localizar señales viales.

In [ ]:
# ── Crear estructura de carpetas YOLO ─────────────────────────────────────

YOLO_DIR = os.path.join(BASE_DIR, 'dataset_yolo')

carpetas_yolo = [
    'images/train', 'images/val', 'images/test',
    'labels/train', 'labels/val', 'labels/test',
]

for carpeta in carpetas_yolo:
    ruta = os.path.join(YOLO_DIR, carpeta)
    os.makedirs(ruta, exist_ok=True)

print('Estructura de carpetas YOLO creada:')
print(f'{YOLO_DIR}/')
for carpeta in carpetas_yolo:
    print(f'  ├── {carpeta}/')
print('  └── data.yaml')

In [ ]:
# ── Generar data.yaml para YOLOv8 ─────────────────────────────────────────

CLASES_YOLO = ['señal_visible', 'señal_deteriorada', 'señal_poco_visible']

data_yaml_contenido = f"""# Dataset YOLO — Señalización Urbana G2-UG-2026
# Generado automáticamente — Avance 2

path: {YOLO_DIR}          # ruta raíz del dataset
train: images/train       # imágenes de entrenamiento
val:   images/val         # imágenes de validación
test:  images/test        # imágenes de prueba

nc: {len(CLASES_YOLO)}    # número de clases
names:
  0: señal_visible
  1: señal_deteriorada
  2: señal_poco_visible

# Descripción de clases
# señal_visible      : señal en buen estado, legible y visible
# señal_deteriorada  : señal con daño físico (óxido, grafiti, deformación)
# señal_poco_visible : señal con baja visibilidad (iluminación, ángulo, obstrucción)
"""

ruta_yaml = os.path.join(YOLO_DIR, 'data.yaml')
with open(ruta_yaml, 'w', encoding='utf-8') as f:
    f.write(data_yaml_contenido)

print(f'data.yaml generado en: {ruta_yaml}')
print('\nContenido del data.yaml:')
print('─' * 50)
print(data_yaml_contenido)

In [ ]:
# ── Visualizar formato de anotación YOLO con ejemplo ──────────────────────
# Muestra cómo se ve una imagen con su bounding box anotado en formato YOLO

def dibujar_bbox_yolo(img_pil, clases_yolo, anotaciones_yolo):
    """
    Dibuja bounding boxes en formato YOLO sobre una imagen.
    anotaciones_yolo: lista de (class_id, x_center, y_center, width, height) — valores [0,1]
    """
    img_np = np.array(img_pil)
    h, w = img_np.shape[:2]

    colores_bbox = [(46, 204, 113), (231, 76, 60), (243, 156, 18)]

    for class_id, xc, yc, bw, bh in anotaciones_yolo:
        x1 = int((xc - bw/2) * w)
        y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w)
        y2 = int((yc + bh/2) * h)
        color = colores_bbox[class_id % len(colores_bbox)]
        cv2.rectangle(img_np, (x1, y1), (x2, y2), color, 3)
        etiqueta = clases_yolo[class_id]
        cv2.putText(img_np, etiqueta, (x1, max(y1-8, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return img_np


# Ejemplo simulado — imagen de la clase 'deteriorada'
ruta_ej = os.path.join(BASE_DIR, 'deteriorada')
if os.path.exists(ruta_ej):
    archivo_ej = sorted([f for f in os.listdir(ruta_ej)
                          if f.lower().endswith(EXT_VALIDAS)])[0]
    img_ej = Image.open(os.path.join(ruta_ej, archivo_ej)).convert('RGB')
    img_ej_resized = img_ej.resize((640, 640))

    # Anotación simulada: una señal deteriorada centrada
    anotacion_demo = [(1, 0.50, 0.48, 0.55, 0.60)]  # class=1 deteriorada

    img_con_bbox = dibujar_bbox_yolo(img_ej_resized, CLASES_YOLO, anotacion_demo)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img_ej_resized)
    axes[0].set_title('Imagen original\n(entrada al anotador)', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_con_bbox)
    axes[1].set_title('Imagen anotada en formato YOLO\n(bounding box señal_deteriorada)',
                       fontweight='bold')
    axes[1].axis('off')

    # Mostrar formato de la etiqueta
    for class_id, xc, yc, bw, bh in anotacion_demo:
        axes[1].text(10, 20, f'Archivo .txt:\n{class_id} {xc:.4f} {yc:.4f} {bw:.4f} {bh:.4f}',
                     fontsize=10, color='white', fontweight='bold',
                     bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

    plt.suptitle('Ejemplo de anotación YOLO — Formato YOLOv8',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RUTA_RESULTADOS, '08_ejemplo_yolo.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

print('Formato YOLO: <class_id> <x_center> <y_center> <width> <height>')
print('Todos los valores normalizados entre 0 y 1 (relativos al tamaño de la imagen)')
print(f'\nHerramienta de anotación: Roboflow (https://roboflow.com)')
print(f'Objetivo: 300 imágenes anotadas (100 por clase) para el Avance 3')

---
## 6. Propuesta de Transformer Visual / Modelo Multimodal

### 6.1 Limitaciones actuales del clasificador CNN/TL
El clasificador MobileNetV2 produce una sola etiqueta por imagen, sin explicar **por qué** clasifica así ni **dónde** está el problema. Los modelos multimodales resuelven esto.

### 6.2 Modelos propuestos

| Modelo | Rol en el sistema | Ventaja para este proyecto |
|---|---|---|
| **CLIP** (OpenAI) | Clasificación por similitud imagen-texto | Puede clasificar sin reentrenar: `"una señal deteriorada con grafiti"` |
| **BLIP** (Salesforce) | Generación de descripciones de imagen | Produce texto describiendo el estado de la señal automáticamente |
| **Swin Transformer** | Clasificación jerárquica | Analiza regiones locales (deterioro parcial) mejor que una CNN |
| **ViT-B/16** | Clasificación global con atención | Complemento experimental a MobileNetV2 |

### 6.3 Integración propuesta en el pipeline final

```
Imagen entrada
     ↓
  YOLOv8 → Detecta y recorta la señal (bounding box)
     ↓
  MobileNetV2 → Clasifica el estado (visible/deteriorada/poco_visible)
     ↓
  BLIP → Genera descripción textual automática del daño
     ↓
  Módulo de diagnóstico → Combina predicción + descripción + métricas → Reporte
```

### 6.4 Por qué BLIP es la elección principal
BLIP-2 puede responder preguntas sobre la imagen (*Visual Question Answering*):
- **Pregunta:** `"¿Qué daño tiene esta señal?"`
- **Respuesta automática:** `"La señal presenta grafiti que cubre más del 60% de la superficie"`

Esto elimina la necesidad de escribir manualmente las descripciones en el módulo de diagnóstico.

In [ ]:
# ── Demo conceptual de CLIP para clasificación zero-shot ─────────────────
# Este bloque instala y prueba CLIP para validar la propuesta

!pip install git+https://github.com/openai/CLIP.git --quiet

import clip
import torch

# Cargar modelo CLIP
clip_model, preprocess_clip = clip.load('ViT-B/32', device=device)
clip_model.eval()

# Textos descriptivos para cada clase del proyecto
textos_clip = [
    "a visible and legible traffic sign in good condition",
    "a deteriorated traffic sign with rust graffiti or physical damage",
    "a traffic sign that is poorly visible due to low light bad angle or obstruction"
]

# Tokenizar textos
tokens_texto = clip.tokenize(textos_clip).to(device)

print('CLIP cargado. Modelo: ViT-B/32')
print(f'Parámetros: {sum(p.numel() for p in clip_model.parameters()):,}')
print('\nTextos de clasificación zero-shot:')
for i, t in enumerate(textos_clip):
    print(f'  [{i}] {t}')

In [ ]:
# ── Clasificación zero-shot con CLIP sobre imágenes de prueba ─────────────
# CLIP no fue entrenado con nuestras imágenes — clasifica por similitud semántica

resultados_clip = []

# Tomar hasta 12 imágenes del conjunto de prueba (4 por clase si hay)
muestra_test = datos_test[:12]

with torch.no_grad():
    for ruta, etiqueta_real in muestra_test:
        img_pil = Image.open(ruta).convert('RGB')
        img_tensor = preprocess_clip(img_pil).unsqueeze(0).to(device)

        # Codificar imagen y texto
        img_features  = clip_model.encode_image(img_tensor)
        text_features = clip_model.encode_text(tokens_texto)

        # Calcular similitud coseno
        img_features  = img_features  / img_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        similitudes   = (100 * img_features @ text_features.T).softmax(dim=-1)

        pred_clip     = similitudes.argmax().item()
        confianza_clip = similitudes[0, pred_clip].item() * 100

        resultados_clip.append({
            'ruta': ruta,
            'real': etiqueta_real,
            'pred': pred_clip,
            'conf': confianza_clip,
            'correcta': pred_clip == etiqueta_real
        })

acc_clip = sum(r['correcta'] for r in resultados_clip) / len(resultados_clip) * 100

print(f'Clasificación zero-shot CLIP sobre {len(muestra_test)} imágenes de prueba:')
print(f'  Accuracy zero-shot: {acc_clip:.1f}%')
print()
print(f'{"Imagen":<35} {"Real":<20} {"Predicho":<20} {"Confianza":>10} {"OK?"}')
print('─' * 95)
for r in resultados_clip:
    nombre = os.path.basename(r['ruta'])[:32]
    real   = CLASES[r['real']]
    pred   = CLASES[r['pred']]
    ok     = '✓' if r['correcta'] else '✗'
    print(f'{nombre:<35} {real:<20} {pred:<20} {r["conf"]:>9.1f}%  {ok}')

In [ ]:
# ── Visualización de predicciones CLIP ───────────────────────────────────

n_mostrar = min(6, len(resultados_clip))
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, r in enumerate(resultados_clip[:n_mostrar]):
    img = Image.open(r['ruta']).convert('RGB')
    axes[i].imshow(img)
    color_border = '#2ecc71' if r['correcta'] else '#e74c3c'
    for spine in axes[i].spines.values():
        spine.set_edgecolor(color_border)
        spine.set_linewidth(4)
    titulo = (f"Real: {CLASES[r['real']]}\n"
              f"CLIP: {CLASES[r['pred']]} ({r['conf']:.1f}%)\n"
              f"{'✓ Correcto' if r['correcta'] else '✗ Error'}")
    axes[i].set_title(titulo, fontsize=9,
                      color=color_border, fontweight='bold')
    axes[i].axis('off')

for j in range(n_mostrar, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f'CLIP Zero-Shot — Clasificación sin entrenamiento\nAcc: {acc_clip:.1f}%',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RUTA_RESULTADOS, '09_clip_predicciones.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print(f'\nConclusión: CLIP alcanza {acc_clip:.1f}% sin ningún entrenamiento en nuestro dataset.')
print('En el Avance 3, se evaluará BLIP para generación de descripciones textuales automáticas.')

---
## 7. Diagnóstico Textual Inicial

El módulo de diagnóstico toma la predicción del modelo y genera una descripción estructurada
del estado de la señal, incluyendo nivel de severidad, evidencia visual y recomendación de acción.

### Plantilla de diagnóstico
```
La imagen fue clasificada como: [clase predicha]
El nivel de severidad estimado es: [bajo / medio / alto / crítico]
La evidencia visual principal es: [descripción del daño o condición observada]
Recomendación: [acción sugerida]
```

In [ ]:
# ── Módulo de diagnóstico textual ─────────────────────────────────────────

def calcular_metricas_imagen(ruta_o_pil):
    """Calcula métricas de calidad de imagen (brillo, contraste, nitidez)."""
    if isinstance(ruta_o_pil, str):
        img_pil = Image.open(ruta_o_pil).convert('RGB')
    else:
        img_pil = ruta_o_pil

    img_np  = np.array(img_pil)
    img_gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

    brillo    = float(img_gray.mean())
    contraste = float(img_gray.std())
    nitidez   = float(cv2.Laplacian(img_gray, cv2.CV_64F).var())

    return brillo, contraste, nitidez


# ── Plantillas de diagnóstico por clase ───────────────────────────────────

DIAGNOSTICOS = {
    'visible_legible': {
        'severidad':   'Bajo',
        'color_sev':   '#2ecc71',
        'evidencias':  [
            'La señal presenta colores vivos y bordes nítidos.',
            'La superficie está limpia y el texto/símbolo es claramente legible.',
            'La iluminación y el ángulo de captura son adecuados.',
        ],
        'recomendacion': 'Mantenimiento preventivo estándar. '
                         'Revisión periódica en 12 meses.',
    },
    'deteriorada': {
        'severidad':   'Alto',
        'color_sev':   '#e74c3c',
        'evidencias':  [
            'La señal presenta daño físico visible: óxido, grafiti, deformación o pintura borrada.',
            'La legibilidad del símbolo o texto está comprometida.',
            'El deterioro físico es intrínseco a la señal, no un artefacto de la imagen.',
        ],
        'recomendacion': 'Intervención urgente: reemplazo o restauración inmediata. '
                         'Registrar ubicación GPS para orden de trabajo.',
    },
    'poco_visible': {
        'severidad':   'Medio',
        'color_sev':   '#f39c12',
        'evidencias':  [
            'La señal no es fácilmente legible por condiciones externas.',
            'Posibles causas: baja iluminación, ángulo inadecuado, vegetación, distancia excesiva.',
            'La señal puede estar en buen estado pero su visibilidad está reducida.',
        ],
        'recomendacion': 'Intervención correctiva: mejorar iluminación, podar vegetación '
                         'o reposicionar la señal. Revisión en campo recomendada.',
    }
}

UMBRALES = {
    'brillo_bajo':     50,
    'brillo_alto':     210,
    'contraste_bajo':  20,
    'nitidez_baja':    50,
}


def generar_diagnostico(ruta_imagen, modelo, transform_eval, clases, device):
    """
    Genera un diagnóstico textual completo para una imagen de señal vial.

    Args:
        ruta_imagen : str — ruta a la imagen
        modelo      : modelo PyTorch entrenado
        transform_eval : transformaciones de evaluación
        clases      : lista de nombres de clases
        device      : dispositivo de cómputo

    Returns:
        dict con clase, confianza, métricas y texto de diagnóstico
    """
    # ── Preprocesar imagen ────────────────────────────────────────────────
    img_pil = Image.open(ruta_imagen).convert('RGB')
    tensor  = transform_eval(img_pil).unsqueeze(0).to(device)

    # ── Predicción ────────────────────────────────────────────────────────
    modelo.eval()
    with torch.no_grad():
        logits       = modelo(tensor)
        probabilidades = F.softmax(logits, dim=1)[0]

    pred_idx    = probabilidades.argmax().item()
    confianza   = probabilidades[pred_idx].item() * 100
    clase_pred  = clases[pred_idx]

    # ── Métricas de imagen ────────────────────────────────────────────────
    brillo, contraste, nitidez = calcular_metricas_imagen(ruta_imagen)

    # ── Alertas de calidad ────────────────────────────────────────────────
    alertas = []
    if brillo < UMBRALES['brillo_bajo']:
        alertas.append(f'⚠ Brillo muy bajo ({brillo:.1f}/255): posible captura nocturna o con sombras.')
    elif brillo > UMBRALES['brillo_alto']:
        alertas.append(f'⚠ Sobreexposición ({brillo:.1f}/255): imagen saturada de luz.')
    if contraste < UMBRALES['contraste_bajo']:
        alertas.append(f'⚠ Contraste bajo ({contraste:.1f}): colores de la señal poco diferenciados.')
    if nitidez < UMBRALES['nitidez_baja']:
        alertas.append(f'⚠ Imagen borrosa (nitidez={nitidez:.1f}): revisar enfoque o distancia de captura.')

    # ── Construir diagnóstico ─────────────────────────────────────────────
    info = DIAGNOSTICOS[clase_pred]
    probs_str = ' | '.join(
        f'{clases[i]}: {probabilidades[i]*100:.1f}%' for i in range(len(clases))
    )

    return {
        'clase':        clase_pred,
        'confianza':    confianza,
        'severidad':    info['severidad'],
        'evidencias':   info['evidencias'],
        'recomendacion': info['recomendacion'],
        'metricas':     {'brillo': brillo, 'contraste': contraste, 'nitidez': nitidez},
        'alertas':      alertas,
        'probabilidades': probs_str,
        'imagen_pil':   img_pil,
    }


def imprimir_diagnostico(d):
    """Imprime el diagnóstico en formato de reporte textual."""
    iconos = {'Alto': '🔴', 'Medio': '🟡', 'Bajo': '🟢'}
    print('╔' + '═' * 68 + '╗')
    print('║  DIAGNÓSTICO DE SEÑAL VIAL — Sistema G2-UG-2026' + ' ' * 19 + '║')
    print('╠' + '═' * 68 + '╣')
    print(f'║  La imagen fue clasificada como : {d["clase"].upper():<33} ║')
    print(f'║  Confianza del modelo          : {d["confianza"]:>5.1f}%{" " * 26} ║')
    print(f'║  Nivel de severidad estimado   : {iconos[d["severidad"]]} {d["severidad"]:<30} ║')
    print('╠' + '═' * 68 + '╣')
    print('║  Probabilidades por clase:' + ' ' * 42 + '║')
    for clase_n, prob in zip(CLASES, d['probabilidades'].split(' | ')):
        val = float(prob.split(': ')[1].replace('%', ''))
        barra = '█' * int(val / 5)
        print(f'║    {clase_n:<20} {val:>5.1f}%  {barra:<14} ║')
    print('╠' + '═' * 68 + '╣')
    print('║  Evidencia visual principal:' + ' ' * 40 + '║')
    for ev in d['evidencias']:
        # Wrap a 62 caracteres
        palabras = ev.split()
        linea = '    •  '
        for palabra in palabras:
            if len(linea) + len(palabra) + 1 > 67:
                print(f'║  {linea:<66} ║')
                linea = '       ' + palabra + ' '
            else:
                linea += palabra + ' '
        if linea.strip():
            print(f'║  {linea:<66} ║')
    print('╠' + '═' * 68 + '╣')
    print('║  Métricas de calidad de imagen:' + ' ' * 37 + '║')
    m = d['metricas']
    print(f'║    Brillo    : {m["brillo"]:>6.1f} / 255' + ' ' * 38 + '║')
    print(f'║    Contraste : {m["contraste"]:>6.1f}' + ' ' * 46 + '║')
    print(f'║    Nitidez   : {m["nitidez"]:>6.1f}' + ' ' * 46 + '║')
    if d['alertas']:
        print('╠' + '═' * 68 + '╣')
        print('║  Alertas de calidad:' + ' ' * 48 + '║')
        for alerta in d['alertas']:
            print(f'║  {alerta[:66]:<66} ║')
    print('╠' + '═' * 68 + '╣')
    print('║  Recomendación:' + ' ' * 53 + '║')
    palabras_r = d['recomendacion'].split()
    linea_r = '    '
    for palabra in palabras_r:
        if len(linea_r) + len(palabra) + 1 > 67:
            print(f'║  {linea_r:<66} ║')
            linea_r = '    ' + palabra + ' '
        else:
            linea_r += palabra + ' '
    if linea_r.strip():
        print(f'║  {linea_r:<66} ║')
    print('╚' + '═' * 68 + '╝')


print('Módulo de diagnóstico textual definido correctamente.')

In [ ]:
# ── Ejecutar diagnóstico sobre imágenes del conjunto de prueba ────────────

print('Generando diagnósticos para imágenes del conjunto de prueba...\n')

# Una imagen de cada clase del test set
muestras_diag = {}
for ruta, etiqueta in datos_test:
    clase_nombre = CLASES[etiqueta]
    if clase_nombre not in muestras_diag:
        muestras_diag[clase_nombre] = ruta
    if len(muestras_diag) == NUM_CLASES:
        break

diagnosticos_resultado = []
for clase_nombre, ruta in muestras_diag.items():
    print(f'─── Diagnóstico para: {clase_nombre} ───')
    d = generar_diagnostico(ruta, modelo_tl, transform_tl_eval, CLASES, device)
    imprimir_diagnostico(d)
    diagnosticos_resultado.append((clase_nombre, ruta, d))
    print()

In [ ]:
# ── Visualización de diagnósticos con imagen ──────────────────────────────

n_diag = len(diagnosticos_resultado)
fig, axes = plt.subplots(1, n_diag, figsize=(6 * n_diag, 7))
if n_diag == 1:
    axes = [axes]

colores_sev = {'Bajo': '#2ecc71', 'Medio': '#f39c12', 'Alto': '#e74c3c', 'Crítico': '#8e44ad'}

for ax, (clase_real, ruta, d) in zip(axes, diagnosticos_resultado):
    ax.imshow(d['imagen_pil'])
    ax.axis('off')

    color_sev = colores_sev.get(d['severidad'], '#95a5a6')
    for spine in ax.spines.values():
        spine.set_edgecolor(color_sev)
        spine.set_linewidth(4)

    titulo = (
        f'Clase real: {clase_real}\n'
        f'Clasificado: {d["clase"]}\n'
        f'Confianza: {d["confianza"]:.1f}%\n'
        f'Severidad: {d["severidad"]}'
    )
    ax.set_title(titulo, fontsize=10, color=color_sev, fontweight='bold')

    # Texto de recomendación debajo
    rec = d['recomendacion'][:70] + ('...' if len(d['recomendacion']) > 70 else '')
    ax.text(0.5, -0.05, rec, transform=ax.transAxes,
            ha='center', va='top', fontsize=8,
            color='#555', wrap=True)

plt.suptitle('Diagnóstico Visual Automático — Una imagen por clase',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_RESULTADOS, '10_diagnosticos_visuales.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Diagnósticos visuales guardados.')

In [ ]:
# ── Plantilla de diagnóstico exportable (formato texto) ───────────────────

PLANTILLA_TEXTO = """
╔══════════════════════════════════════════════════════════════╗
║          PLANTILLA DE DIAGNÓSTICO — SEÑALIZACIÓN VIAL        ║
║              Sistema G2-UG-2026 — Avance 2                   ║
╠══════════════════════════════════════════════════════════════╣
║  La imagen fue clasificada como:    [CLASE PREDICHA]         ║
║  El nivel de severidad estimado es: [BAJO / MEDIO / ALTO]    ║
║  Confianza del modelo:              [X.X%]                   ║
╠══════════════════════════════════════════════════════════════╣
║  La evidencia visual principal es:                           ║
║    [Descripción del daño, obstrucción o condición visual]    ║
╠══════════════════════════════════════════════════════════════╣
║  Métricas de calidad de imagen:                              ║
║    Brillo     : [X.X / 255]                                  ║
║    Contraste  : [X.X]                                        ║
║    Nitidez    : [X.X]                                        ║
╠══════════════════════════════════════════════════════════════╣
║  Recomendación:                                              ║
║    [Acción sugerida basada en la clase y severidad]          ║
╚══════════════════════════════════════════════════════════════╝
"""

ruta_plantilla = os.path.join(RUTA_RESULTADOS, 'plantilla_diagnostico.txt')
with open(ruta_plantilla, 'w', encoding='utf-8') as f:
    f.write(PLANTILLA_TEXTO)

print(PLANTILLA_TEXTO)
print(f'Plantilla guardada en: {ruta_plantilla}')

---
## 8. Resumen y Conclusiones del Avance 2

### 8.1 Logros de este avance

| Objetivo específico | Estado |
|---|---|
| Actualizar y documentar el dataset | ✅ Completado — 240 imágenes, 3 clases, problemas identificados |
| Reproducir CNN baseline del Avance 1 | ✅ Métricas documentadas como línea base |
| Implementar transfer learning | ✅ MobileNetV2 con fine-tuning en 2 fases |
| Comparar baseline vs TL | ✅ Tabla comparativa + matrices de confusión |
| Definir clases y estructura YOLO | ✅ data.yaml + estructura de carpetas creada |
| Proponer Transformer / multimodal | ✅ CLIP zero-shot ejecutado + propuesta BLIP |
| Plantilla de diagnóstico textual | ✅ Módulo funcional con diagnóstico por imagen |

### 8.2 Hallazgos principales
1. **Transfer Learning supera al Baseline**: la brecha entrenamiento-prueba se reduce significativamente
2. **CLIP demuestra viabilidad multimodal**: alcanza accuracy notable sin entrenamiento específico
3. **La confusión persiste** entre `deteriorada` y `poco_visible`: requiere más datos nocturnos
4. **La estructura YOLO está lista** para etiquetado en Roboflow

### 8.3 Plan para el Avance 3 / Presentación Final (21 julio)

| Semana | Actividad |
|---|---|
| Sem. 3–4 | Etiquetar 300 imágenes con bounding boxes en Roboflow + entrenar YOLOv8 |
| Sem. 5 | Experimento con ViT-B/16 como alternativa a MobileNetV2 |
| Sem. 6 | Implementar BLIP para generación de descripciones automáticas |
| Sem. 7 | Integración del pipeline completo: YOLO → Clasificador → Diagnóstico |
| Sem. 8 | Redacción informe final + preparación presentación |

In [ ]:
# ── Resumen final de archivos generados ───────────────────────────────────

print('=' * 65)
print('  AVANCE 2 COMPLETADO — Archivos generados')
print('=' * 65)

if os.path.exists(RUTA_RESULTADOS):
    archivos = sorted(os.listdir(RUTA_RESULTADOS))
    for archivo in archivos:
        ruta_a = os.path.join(RUTA_RESULTADOS, archivo)
        tamanio = os.path.getsize(ruta_a) / 1024
        print(f'  {archivo:<45} {tamanio:>7.1f} KB')

print('─' * 65)
print(f'  Modelo TL guardado: {RUTA_MODELO_TL}')
print(f'  Dataset YOLO listo en: {YOLO_DIR}')
print('=' * 65)
print()
print('  Grupo 2 — Universidad de Guayaquil')
print('  Bajaña · Delgado · Macias · Suarez · Tirado')
print('  Avance 2 — Junio 2026')